# 강의 04 · 실습 6 — 커스텀 MCP 서버 · (5) 고난도 II

## 1. 문제상황

- 쇼핑몰 상담원은 고객이 "무선 마우스 50개 주문해 주세요"라고 하면 재고를 확인하고, 재고가 모자라면 거절하고, 충분하면 재고에서 빼고 접수 번호를 줍니다.
- 지금은 재고 확인과 차감이 서로 다른 화면에 있어서, 확인만 하고 차감을 잊거나 재고가 모자란데 접수해 버리는 일이 생깁니다.
- 주문을 받을 때마다 남은 재고가 줄어야 하므로, 같은 상담 안에서 두 번째 주문은 첫 번째 주문 뒤의 재고를 봐야 합니다.
- 이 판단과 차감을 모델이 도구로 하게 하려면, 재고를 기억하는 변수가 서버 쪽에 있어야 합니다.

## 2. 문제와 목표

- **문제**: 재고 확인과 차감이 따로 있어 잘못된 접수가 생기고, 주문 사이에 재고가 이어지지 않습니다.
- **목표**: 주문을 받아 재고를 검사하고 차감까지 한 도구 안에서 하는 서버 `order`를 만들고, 클라이언트는 서버와의 세션을 하나 열어 둔 채 주문 요청 세 개를 차례로 보내 서버 쪽 재고가 주문마다 이어서 줄어드는 것을 확인합니다.
    - 서버의 도구 두 개: `place_order(item, qty)`는 재고가 모자라면 거절 사유를, 충분하면 재고를 차감하고 접수 번호를 하나 올린 뒤 접수 번호와 남은 재고를 딕셔너리로 돌려줍니다. 돌려주는 딕셔너리에는 성공 여부를 `ok` 키(true/false)에 담습니다. `get_stock(item)`는 현재 재고를 돌려줍니다.
    - 서버 파일 첫 줄은 `from mcp.server.fastmcp import FastMCP`입니다.
    - 처음 재고와 접수 번호: 무선 마우스 37개·키보드 0개·모니터 5개, 접수 번호는 1000 다음부터입니다. 재고 딕셔너리와 카운터는 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - 세션: 서버 프로세스 하나를 열어 두는 방법은 `async with client.session("order") as session:`이고, 그 세션에서 도구 목록을 받는 함수는 `langchain_mcp_adapters.tools`의 `load_mcp_tools(session)`입니다. 두 API는 강의 자료에 없으므로 여기서 제공합니다. 세션을 열어 두지 않으면 도구 호출마다 서버가 새로 떠서 재고가 처음 값으로 돌아갑니다.
    - 접수 번호 카운터: 함수 안에서 함수 밖의 값(접수 번호)을 바꾸려면 파이썬 `global` 선언이 필요합니다. 자세한 내용은 인터넷을 검색해서 익혀봅니다.
    - 주문 요청 세 건(50개·30개·10개)과 확인 질문은 코드에 미리 정해 넣습니다. 도구 목록은 「서버가 준 도구:」 줄로 출력합니다.
- **목표 달성 여부의 판정 기준**
    - 「무선 마우스 50개」 주문은 거절되어 재고 부족 사유가 돌아옵니다.
    - 「무선 마우스 30개」 주문은 접수되어 접수 번호와 남은 재고 7이 돌아옵니다.
    - 이어서 「무선 마우스 10개」 주문은 다시 거절되는 것을 도구 결과 메시지에서 확인합니다. 세 주문이 하나의 세션(같은 서버 프로세스) 위에서 처리되어 재고가 이어집니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex06_s5_diagram.svg)

## 4. 단계별 요구사항

이 단에서는 요구사항을 제공하지 않습니다. 「2. 문제와 목표」의 목표와 「3. 워크플로우 다이어그램」을 보고 요구사항을 직접 번호 목록으로 쓴 뒤 코드를 씁니다.

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

클라이언트 쪽 준비입니다. 라이브러리를 불러오고 모델을 준비하고, 도구 호출 루프 `build_loop`와 연결 선언 함수 `server_config`를 정의합니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

- `build_loop`는 받아 온 도구를 모델에 묶고 model·tools 두 노드를 조건부 엣지로 연결하는 도구 호출 루프입니다. 서버 코드가 아니라 서버를 쓰는 쪽의 코드입니다.
- `sys.stderr = sys.__stderr__` 줄은 노트북 전용입니다. 노트북 커널은 표준 오류 스트림을 화면용 객체로 바꿔 두는데, 서버 프로세스를 띄우는 코드는 원래의 표준 오류 스트림을 요구하므로 되돌려 놓습니다. 이 줄은 MCP 클라이언트를 불러오기 전에 있어야 합니다.
- `server_config`는 서버 파일 하나를 표준입출력으로 띄우는 연결 선언입니다. `sys.executable`은 지금 돌고 있는 파이썬 러너입니다. `FASTMCP_LOG_LEVEL`은 서버의 안내 로그가 화면을 채우지 않게 하는 설정입니다.
- 실행 결과 출력은 `show(result)`, 메시지 글자 추출은 `text_of(m)`로 합니다.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

sys.stderr = sys.__stderr__   # 노트북 커널의 stderr에는 fileno()가 없어 서버 프로세스 시작이 실패하므로 원래 stderr로 되돌린다
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")


class State(TypedDict):
    messages: Annotated[list, add_messages]


def build_loop(tools):
    """도구 호출 루프. 도구를 모델에 묶고 model·tools 두 노드를 조건부 엣지로 연결한다."""
    bound = llm.bind_tools(tools)

    def call_model(state: State) -> dict:
        return {"messages": [bound.invoke(state["messages"])]}

    def should_continue(state: State) -> str:
        return "tools" if state["messages"][-1].tool_calls else END

    g = StateGraph(State)
    g.add_node("model", call_model)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "model")
    g.add_conditional_edges("model", should_continue, {"tools": "tools", END: END})
    g.add_edge("tools", "model")
    return g.compile()


def text_of(m) -> str:
    """메시지 내용이 콘텐츠 블록 목록이면 글자 부분만 이어 붙인다."""
    if isinstance(m.content, list):
        return " ".join(p.get("text", "") for p in m.content if isinstance(p, dict))
    return str(m.content)


def show(result) -> None:
    """실행 결과의 메시지를 종류·도구 호출·상태와 함께 한 줄씩 출력한다."""
    for m in result["messages"]:
        kind = type(m).__name__
        calls = getattr(m, "tool_calls", None)
        if calls:
            print(f"[{kind}] tool_calls={[(c['name'], c['args']) for c in calls]}")
        elif kind == "ToolMessage":
            print(f"[{kind}] status={m.status!r} {text_of(m)[:160]}")
        elif m.content:
            print(f"[{kind}] {text_of(m)[:300]}")


def server_config(file: str) -> dict:
    """서버 파일 하나를 표준입출력으로 띄우는 연결 선언을 만든다."""
    return {"command": sys.executable, "args": [str(Path(file).resolve())],
            "transport": "stdio", "env": {"FASTMCP_LOG_LEVEL": "ERROR"}}


print("클라이언트 준비를 마쳤습니다.")

# 주어진 자료 — 서버 파일(order_server.py)의 처음 재고와 접수 번호 카운터. 서버 파일의 도구 등록 코드에 그대로 옮겨 적는다
STOCK = {"무선 마우스": 37, "키보드": 0, "모니터": 5}   # 서버 프로세스가 살아 있는 동안 유지된다
ORDER_NO = 1000


In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다. 서버 파일은 `%%writefile` 셀로 만듭니다.


## 7. 실행 결과 확인

위 실행 결과에서 다음을 확인합니다.

1. `서버가 준 도구:` 줄에 `place_order`와 `get_stock`이 설명과 함께 있습니다.
2. 첫 요청(50개)의 `ToolMessage`에 `"ok": false`와 재고 부족 사유가 있고, 모델이 거절을 답합니다.
3. 둘째 요청(30개)의 `ToolMessage`에 `"ok": true`, 접수 번호, 남은 재고 7이 있습니다.
4. 셋째 요청(10개)의 `ToolMessage`에 다시 `"ok": false`가 있습니다. 둘째 요청이 차감한 재고가 이어졌기 때문입니다. 마지막 `get_stock` 확인에서 7이 나옵니다.